# 4.6 · 弹性网 / Elastic Net

> **课程定位 / Where this fits**
> **Part 4 第 6 课, 正则化三部曲收官**。Ridge(稳定但不稀疏) + Lasso(稀疏但共线下任性) → Elastic Net **同时用 L1+L2**, 取两者之长。本课讲清它解决了 Lasso 的哪两个痛点, 以及 `l1_ratio` 怎么调。
> The finale of the regularization trio: L1+L2 combined, fixing Lasso's two pain points.

> 💡 **面试相关 / Interview-relevant**
> - "Elastic Net 解决 Lasso 什么问题" ★★★★
> - "l1_ratio 是什么" ★★★
> - "什么时候用 Elastic Net 而非 Lasso/Ridge" ★★★★（高维共线 + n<p）

---

## 学习目标 / Learning Objectives
1. 写出 Elastic Net 目标, 理解 L1+L2 混合。
2. 理解它修复 Lasso 的**两个痛点**：共线"任选一个" + $n<p$ 时最多选 $n$ 个特征。
3. 用 `l1_ratio` 在 Ridge↔Lasso 之间连续滑动。
4. 双超参 CV（α 和 l1_ratio）。

## 目录 / TOC
1. [目标函数: L1+L2 混合 ⭐](#1)
2. [修复 Lasso 的两个痛点 ⭐](#2)
3. [数据](#3)
4. [l1_ratio: Ridge↔Lasso 滑块](#4)
5. [共线分组效应](#5)
6. [双超参 CV](#6)
7. [三部曲总对比 + 选择决策](#7)
8. [小结](#8)


<a id="1"></a>
## 1. 目标函数: L1+L2 混合 ⭐ / Objective

$$J(\mathbf{w}) = \|\mathbf{y}-\mathbf{X}\mathbf{w}\|^2 + \lambda\Big(\underbrace{\alpha_1 \|\mathbf{w}\|_1}_{\text{L1: 稀疏}} + \underbrace{\alpha_2 \|\mathbf{w}\|_2^2}_{\text{L2: 稳定}}\Big)$$

sklearn 参数化（`ElasticNet(alpha, l1_ratio)`）：
$$J = \frac{1}{2n}\|\mathbf{y}-\mathbf{X}\mathbf{w}\|^2 + \alpha\Big(\rho\|\mathbf{w}\|_1 + \frac{1-\rho}{2}\|\mathbf{w}\|_2^2\Big)$$
- `alpha` ($\alpha$) = 总正则强度
- `l1_ratio` ($\rho$) = L1 占比: **ρ=1 纯 Lasso, ρ=0 纯 Ridge**, 中间是混合

→ Elastic Net 把 Ridge 和 Lasso 都变成它的**特例**, 用一个连续旋钮统一。
Elastic Net makes both Ridge and Lasso special cases on a single continuous dial.


<a id="2"></a>
## 2. 修复 Lasso 的两个痛点 ⭐ / Fixing Lasso's Two Pains

| Lasso 痛点 | Elastic Net 的修复 |
|---|---|
| **共线特征任选其一**（4.5: 一组高度相关特征里 Lasso 随机留一个, 其余归 0, 不稳定）| L2 项带来**分组效应**: 共线特征**一起被选或一起被压**, 系数平摊, 稳定 |
| **$n < p$ 时最多选 $n$ 个特征**（基因数据 p=2万 n=200, Lasso 上限选 200 个）| L2 项**解除这个上限**, 可选超过 $n$ 个特征 |

**直觉**: L1 给稀疏, L2 给稳定 + 分组。**两个世界最好的部分**。
L1 brings sparsity, L2 brings stability + grouping. The best of both.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(42)

data = fetch_california_housing(as_frame=True)
X, y = data.data.values, data.target.values
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0)
scaler = StandardScaler().fit(X_tr)
Xtr, Xte = scaler.transform(X_tr), scaler.transform(X_te)
print(f"{X.shape}")


<a id="4"></a>
## 4. l1_ratio: Ridge↔Lasso 滑块 / The Slider


In [ ]:
from sklearn.linear_model import ElasticNet

# 固定 alpha, 扫 l1_ratio: 看系数从 Ridge 式(都非零) 到 Lasso 式(部分归零)
ratios = [0.0, 0.2, 0.5, 0.8, 1.0]
print(f"{'l1_ratio':<10} {'非零系数数':<12} 说明")
for r in ratios:
    # l1_ratio=0 时 sklearn 要求用 Ridge, 这里用极小值近似 / ~Ridge at 0
    en = ElasticNet(alpha=0.1, l1_ratio=max(r, 0.001), max_iter=5000).fit(Xtr, y_tr)
    nz = (np.abs(en.coef_) > 1e-6).sum()
    tag = "≈Ridge(全保留)" if r==0 else ("纯Lasso(最稀疏)" if r==1 else "混合")
    print(f"{r:<10} {nz}/{X.shape[1]:<11} {tag}")
print("\nl1_ratio 从 0→1: 越来越像 Lasso, 归零的系数越多")


<a id="5"></a>
## 5. 共线分组效应 / Grouping Effect on Collinear Features

复现 4.5 的痛点: 一组强共线特征。看 Lasso(任选一个) vs Elastic Net(一起保留)。


In [ ]:
# 造 3 个强共线特征 + 真实信号 / three collinear features
n = 300
base = rng.normal(0, 1, n)
x1 = base + rng.normal(0, 0.05, n)
x2 = base + rng.normal(0, 0.05, n)
x3 = base + rng.normal(0, 0.05, n)     # x1,x2,x3 几乎相同
x4 = rng.normal(0, 1, n)                # 独立无关特征
y_syn = 3*base + rng.normal(0, 0.5, n)  # 真实依赖 base (= x1≈x2≈x3)
Xc = np.c_[x1, x2, x3, x4]

from sklearn.linear_model import Lasso
print("真实: y=3·base, 而 x1≈x2≈x3≈base (共线组), x4 无关\n")
lasso = Lasso(alpha=0.1).fit(Xc, y_syn)
en = ElasticNet(alpha=0.1, l1_ratio=0.5).fit(Xc, y_syn)
print(f"{'':12} {'x1':>7} {'x2':>7} {'x3':>7} {'x4':>7}")
print(f"{'Lasso':<12} " + " ".join(f"{c:>7.2f}" for c in lasso.coef_))
print(f"{'ElasticNet':<12} " + " ".join(f"{c:>7.2f}" for c in en.coef_))
print("\nLasso: 共线组里只留 1-2 个, 其余归0 (任性, 换数据可能换个留)")
print("ElasticNet: 把系数平摊给 x1,x2,x3 (分组效应, 稳定); x4 正确归0")


<a id="6"></a>
## 6. 双超参 CV / Two-hyperparameter CV

Elastic Net 有**两个**超参: `alpha`(强度) 和 `l1_ratio`(L1占比)。`ElasticNetCV` 在二维网格上搜。


In [ ]:
from sklearn.linear_model import ElasticNetCV

encv = ElasticNetCV(l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 1.0],
                    cv=5, max_iter=10000, random_state=0).fit(Xtr, y_tr)
print(f"ElasticNetCV 选出: alpha={encv.alpha_:.4f}, l1_ratio={encv.l1_ratio_}")
print(f"test R² = {encv.score(Xte, y_te):.4f}")
print(f"\n非零系数: {(np.abs(encv.coef_)>1e-6).sum()}/{X.shape[1]}")
print("l1_ratio 也是被 CV 选出来的 → 数据自己决定要多少稀疏 vs 多少稳定")


<a id="7"></a>
## 7. 三部曲总对比 + 选择决策 ⭐ / The Trio

```
普通 OLS  →  无正则, 易过拟合/共线不稳
  ├── Ridge (L2):     系数收缩不归零, 稳定, 治共线 (平摊), 保留全特征
  ├── Lasso (L1):     系数精确归零, 稀疏/特征选择, 共线下任选其一, n<p 上限选n个
  └── Elastic Net:    L1+L2 混合, 稀疏 + 共线分组 + 解除 n<p 上限 ⭐
                      l1_ratio: 0=Ridge ←→ 1=Lasso 连续滑动

选择决策:
  特征都重要, 只想防过拟合/治共线   → Ridge
  想自动特征选择, 特征间不太共线     → Lasso
  高维 + 共线 + 想要稀疏 (如基因/文本) → Elastic Net (最稳妥的默认)
  不确定 → ElasticNetCV (它会自己滑到合适位置, Ridge/Lasso 都是它特例)
```


In [ ]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso

# 三部曲在同一数据上的横向对比 (加噪声特征放大差异) / head-to-head
noise = rng.normal(size=(len(X), 30))
Xa = np.c_[X, noise]
Xa_tr, Xa_te, ya_tr, ya_te = train_test_split(Xa, y, test_size=0.3, random_state=0)
sc = StandardScaler().fit(Xa_tr)
Xa_tr, Xa_te = sc.transform(Xa_tr), sc.transform(Xa_te)

models = {
    "OLS": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=0.05, max_iter=10000),
    "ElasticNet": ElasticNet(alpha=0.05, l1_ratio=0.5, max_iter=10000),
}
print(f"8 真实 + 30 噪声特征:")
print(f"{'模型':<12} {'test R²':>9} {'非零系数':>9}")
for name, m in models.items():
    m.fit(Xa_tr, ya_tr)
    nz = (np.abs(m.coef_) > 1e-6).sum() if hasattr(m, "coef_") else "-"
    print(f"{name:<12} {m.score(Xa_te, ya_te):>9.4f} {nz:>9}/38")
print("\nLasso/ElasticNet 剔除大量噪声 → 更简洁; 噪声越多正则优势越明显")


<a id="8"></a>
## 8. 小结 / Summary

```
Elastic Net = L1 + L2 混合, l1_ratio 在 Ridge(0)↔Lasso(1) 滑动
修复 Lasso 两痛点:
  共线"任选一个" → L2 分组效应, 共线特征一起选/压, 稳定
  n<p 最多选 n 个 → L2 解除上限
双超参 (alpha, l1_ratio), ElasticNetCV 二维搜
正则三部曲: Ridge(稳定全留) / Lasso(稀疏选择) / ElasticNet(两者兼得)
```

### 💡 面试速查
1. **Elastic Net = L1+L2**, 修 Lasso 的共线任性 + n<p 上限
2. **分组效应**: 共线特征被一起选/压 (Lasso 随机留一个)
3. **l1_ratio**: 0=Ridge, 1=Lasso, Elastic Net 是统一框架
4. **高维共线场景默认 Elastic Net**
5. **三部曲都需标准化 + 都不惩罚偏置**

### 下一节
**4.7 广义线性模型 GLM**——前 6 课目标 y 都假设高斯。但计数(泊松)、二值、保险理赔(伽马)呢？GLM 用 link function 把线性模型推广到指数族。
